In [ ]:
def compute_total_response_time(csv_filename: str) -> tuple[float, int]:
    '''Computes the total response time and the number of responses from a CSV file.

    This function reads a CSV file where each row contains response times recorded for a given client in seconds. 
    It calculates the total sum of all response times across all rows.

    Parameters:
    -----------
    csv_filename : str
        The path to the CSV file containing response times. Each row represents a set of response times
        separated by commas.

    Returns:
    --------
    tuple[float, int]
        A tuple where:
        - The first element (float) is the total sum of all response times.
        - The second element (int) is the number of responses.
    '''

    total_response_time = 0.0
    number_of_responses = 0

    with open(csv_filename, 'r', newline='') as response_time_csv:
        system_response_times = csv.reader(response_time_csv, delimiter=',')

        for client_response_times in system_response_times:
            for response_time in client_response_times:
                total_response_time += float(response_time)
                number_of_responses += 1

    return total_response_time, number_of_responses

def compute_average_response_time() -> float:
    total_response_time, number_of_responses = compute_total_response_time(".data/csv/response_time.csv")
    return total_response_time/number_of_responses



: 

In [ ]:
class DelayAnalyzer:
    def __init__(self, file_path):
        self.file_path = file_path

    def mean_mu_observed(self, mu):
        # Retrieve all the delays observation and eveluate the mean. Also return the number of observed values
        data = pd.read_csv(self.file_path)
        filtered_data = data[data['mu'] == mu]
        n = filtered_data.shape[0]
        if not filtered_data.empty:
            mean_delay = filtered_data['delay'].mean() 
            return 1 / mean_delay, n  # Return the rate (1/mean delay)
        else:
            raise ValueError(f"No data found for mu = {mu}")

    def empirical_distribution(self,mu):
        # return all the delays
        data = pd.read_csv(self.file_path)
        filtered_data = data[data['mu'] == mu]
        return filtered_data['delay']
    
    def log_delay_to_csv(self, mu, delay):
        # Log the delay into a CSV file.
        file_exists = os.path.isfile(self.file_path)

        # Open the file in append mode, create it if it doesn't exist
        with open(self.file_path, mode='a', newline='') as file:
            writer = csv.writer(file)
            if not file_exists:
                writer.writerow(['mu', 'delay'])  # Writing the header
            writer.writerow([mu, delay])  # Writing the mu and delay values


class PlotGenerator:
    def __init__(self, image_path):
        self.file_path = image_path

    def generate_exponential_plot(self, mu, mean_mu, values, n):
        # int(10/mu) + np.log10(mu) in order to have enogh space to represent all data
        # int(10/mu) guarantee that lower values have a correct representation when np.log10(mu) will be 0
        # otherwise if the mu value is higher, np.log10(mu) will be reasonable and int(10/mu) will be zero
        
        time_values = np.linspace(0, int(10/mu) + np.log10(mu), 1000)
        pdf = mu * np.exp(-mu * time_values)

        plt.figure(figsize=(8, 6))
        plt.hist(values,bins=np.linspace(0, int(10/mu)  + np.log10(mu), 30), density=True)
        plt.plot(time_values, pdf, label=f'PDF (mu = {mu:.2f})')

        plt.xlabel('Time (seconds)')
        plt.ylabel('Probability Density')
        plt.title(f'Exponential Distribution with Rate {mu} vs {mean_mu:.2f} with {n} trials')
        plt.legend()
        plt.grid(True)
        plt.savefig(self.file_path) # check this becoause the path isn't consistent and loclahost:5000/plot raise a 500 server error
        plt.close()
